# Ex3_homogeneous_multigroup

A homogeneous four group slab with energy coupling and power iterations.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

working_directory = Path.cwd().resolve()
candidates = (
    working_directory, *working_directory.parents,
    working_directory / 'Ex_RP_SN_discrete_ordinates',
    working_directory / 'RPTT_PoliTO_GitHub' / 'Ex_RP_SN_discrete_ordinates',
)
PROJECT_DIRECTORY = next(
    (path for path in candidates if (path / 'sn_1d.py').is_file()), None
)
if PROJECT_DIRECTORY is None:
    raise FileNotFoundError('Cannot find sn_1d.py.')

sys.path.insert(0, str(PROJECT_DIRECTORY))
from sn_1d import SNSettings, SNSolver, SlabMesh, SlabSNProblem

DATA_DIRECTORY = PROJECT_DIRECTORY / 'data'
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True})
print(f'Input: {DATA_DIRECTORY}')


In [ ]:
def plot_inner_iteration_maps(result):
    # One dot per (power iteration, outer iteration) pair.
    power_x, outer_y, per_outer, per_power = [], [], [], []
    for power in result.history:
        counts = [sum(inner.iterations for inner in outer.inner_iterations)
                  for outer in power.outer_iterations]
        total = sum(counts)
        for outer, count in zip(power.outer_iterations, counts):
            power_x.append(power.iteration)
            outer_y.append(outer.iteration)
            per_outer.append(count)
            per_power.append(total)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
    for ax, values, title, colorbar_label in (
        (axes[0], per_power, 'Total per power iteration (same color in each column)',
         'total inner iterations in this power iteration'),
        (axes[1], per_outer, 'Count per outer iteration (color of each dot)',
         'inner iterations at this (power, outer) pair'),
    ):
        dots = ax.scatter(power_x, outer_y, c=values, cmap='viridis',
                          s=75, edgecolors='black', linewidths=0.3)
        fig.colorbar(dots, ax=ax, label=colorbar_label)
        ax.set(xlabel='Power iteration', ylabel='Outer iteration', title=title)
        ax.set_yticks(range(1, max(outer_y) + 1))
    fig.tight_layout()
    return fig, axes


def plot_angular_and_scalar_flux(result, group=0, normalize=True):
    # Angular and scalar flux for the selected group on the same axes.
    if not 0 <= group < result.scalar_flux.shape[0]:
        raise ValueError('GROUP_TO_PLOT is out of range.')
    x = result.problem.mesh.cell_centers
    angular = result.angular_flux[group].copy()
    scalar = result.scalar_flux[group].copy()
    if normalize:
        scale = np.max(np.abs(scalar))
        angular /= scale
        scalar /= scale
        ylabel = 'flux / maximum scalar flux of the group'
    else:
        ylabel = 'flux'
    signed_mu = np.r_[result.directions, -result.directions]
    fig, ax = plt.subplots()
    for ordinate, mu in enumerate(signed_mu):
        ax.plot(x, angular[ordinate], lw=1, alpha=0.75,
                label=fr'$\psi(\mu={mu:+.3f})$')
    ax.plot(x, scalar, 'k', lw=2.8, label=fr'$\phi_{{{group + 1}}}$')
    for interface in result.problem.mesh.region_edges[1:-1]:
        ax.axvline(interface, color='0.6', ls='--', lw=0.8)
    ax.set(xlabel='$z$ [cm]', ylabel=ylabel,
           title=f'Angular and scalar flux — group {group + 1}')
    ax.legend(ncol=2, fontsize=8)
    return fig, ax


def remesh_problem(problem, target_width):
    # Preserve geometry and materials while changing the spatial mesh.
    lengths = np.diff(problem.mesh.region_edges)
    cells = np.maximum(1, np.rint(lengths / target_width).astype(int))
    mesh = SlabMesh(problem.mesh.region_edges.copy(), cells)
    return SlabSNProblem(mesh, problem.cross_sections,
                         name=f'{problem.name}, dz~{target_width:g} cm')


def keff_parameter_study(base_problem, orders, target_widths,
                         use_previous_flux=True):
    values = np.empty((len(target_widths), len(orders)))
    cell_counts = np.empty(len(target_widths), dtype=int)
    for row, width in enumerate(target_widths):
        problem = remesh_problem(base_problem, width)
        cell_counts[row] = problem.mesh.number_of_cells
        for col, order in enumerate(orders):
            settings = SNSettings(
                order=order, inner_tolerance=1e-4, outer_tolerance=5e-4,
                eigenvalue_tolerance=1e-4,
                multigroup_warm_start=use_previous_flux,
            )
            values[row, col] = SNSolver(problem, settings).solve_critical().keff
    return values, cell_counts


## Initial guess for inner iterations

Set `USE_PREVIOUS_FLUX_AS_INNER_GUESS` to `True` to reuse the current
angular flux. On the first outer iteration of a power step, this is the
angular flux from the preceding power step. Set it to `False` to restart
every group solve from unit angular flux. In the fixed source problem,
the same switch controls reuse between outer iterations.


In [ ]:
USE_PREVIOUS_FLUX_AS_INNER_GUESS = True

settings = SNSettings(
    order=4, inner_tolerance=1e-05,
    outer_tolerance=0.0001, eigenvalue_tolerance=1e-05,
    multigroup_warm_start=USE_PREVIOUS_FLUX_AS_INNER_GUESS,
)
solver = SNSolver.from_case(
    DATA_DIRECTORY, case='homogeneous', multigroup=True, settings=settings
)
result = solver.solve_critical()

print(f'k_eff = {result.keff:.8f}')
print(f'power iterations = {result.power_iterations}')
print(f'groups = {result.scalar_flux.shape[0]}')
print(f'materials = {result.problem.cross_sections.region_materials}')


## Nested iteration counts

All completed iterations are plotted automatically. A dot at `(3, 2)`
means **outer iteration 2 of power iteration 3**. The right panel
colors that dot by the number of inner iterations needed for that
outer iteration (summed over energy groups). The left panel colors
every dot in column 3 by the **total** inner iterations over all
outer iterations of power iteration 3. For a one group case, only
outer iteration 1 occurs.


In [ ]:
plot_inner_iteration_maps(result);


## Angular and scalar flux

Select a group with a zero based Python index. Both fluxes are divided
by the maximum scalar flux of that group, preserving their relative
magnitudes.


In [ ]:
GROUP_TO_PLOT = 3
plot_angular_and_scalar_flux(result, GROUP_TO_PLOT);


## Angular and spatial convergence of $k_{eff}$

Both panels report $(k-k_{\mathrm{finest}})\times10^5$ pcm, where
$k_{\mathrm{finest}}$ is computed at the largest angular order and
smallest target mesh width in the table below. Left: change $N$ for
each fixed mesh. Right: change the spatial mesh for each fixed $N$.
The number of energy groups and all material data stay fixed.


In [ ]:
# Extend these lists for a more detailed (and more expensive) study.
ORDERS = (2, 4, 6)                    # Even angular orders
TARGET_MESH_WIDTHS = (4.0, 2.0, 1.0)  # Target cell widths in cm

keff_values, spatial_cells = keff_parameter_study(
    solver.problem, ORDERS, TARGET_MESH_WIDTHS,
    use_previous_flux=USE_PREVIOUS_FLUX_AS_INNER_GUESS,
)

reference = keff_values[np.argmin(TARGET_MESH_WIDTHS), np.argmax(ORDERS)]
delta_pcm = (keff_values - reference) * 1e5
print(f'Finest reference k_eff = {reference:.8f}')
print('Rows: spatial mesh; columns: angular order N; values: delta k [pcm]')
print('N =', ORDERS)
for index, (width, cells) in enumerate(zip(TARGET_MESH_WIDTHS, spatial_cells)):
    print(f'{cells:3d} cells (target dz~{width:g} cm):',
          ' '.join(f'{value:+.1f}' for value in delta_pcm[index]))

fig, (ax_order, ax_mesh) = plt.subplots(1, 2, figsize=(12, 4.5))
for row, cells in enumerate(spatial_cells):
    ax_order.plot(ORDERS, delta_pcm[row], 'o-', label=f'{cells} cells')
ax_order.set(xlabel='Angular order N', ylabel=r'$(k-k_{\mathrm{finest}})\times10^5$ [pcm]',
             title='Angular order at fixed mesh', xticks=ORDERS)
ax_order.legend()

for col, order in enumerate(ORDERS):
    ax_mesh.plot(spatial_cells, delta_pcm[:, col], 'o-', label=fr'$S_{order}$')
ax_mesh.set(xlabel='Total number of cells', ylabel=r'$(k-k_{\mathrm{finest}})\times10^5$ [pcm]',
            title='Spatial mesh at fixed angular order')
ax_mesh.legend()
fig.tight_layout();
